# 02 - Data Preprocessing for PyTorch

This notebook prepares the UCI HAR dataset for deep learning.

Objectives:
- Inspect the existing sensor signal distributions.
- Understand the participant-wise train/test split.
- Convert activity labels into zero-based indices.
- Prepare signals for a PyTorch 1D CNN.
- Build PyTorch Dataset and DataLoader objects.

## Imports and load existing data

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

from paths import (
    TRAIN_SIGNALS_DIR,
    TEST_SIGNALS_DIR,
    TRAIN_LABELS_FILE,
    TEST_LABELS_FILE,
    TRAIN_SUBJECTS_FILE,
    TEST_SUBJECTS_FILE,
)

CHANNELS = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
]

def load_signals(signals_dir, split):
    channel_data = []

    for channel in CHANNELS:
        file_path = signals_dir / f"{channel}_{split}.txt"
        data = np.loadtxt(file_path)
        channel_data.append(data)

    return np.stack(channel_data, axis=-1)


def load_labels(labels_file):
    return np.loadtxt(labels_file, dtype=int)


X_train = load_signals(TRAIN_SIGNALS_DIR, "train")
y_train = load_labels(TRAIN_LABELS_FILE)

X_test = load_signals(TEST_SIGNALS_DIR, "test")
y_test = load_labels(TEST_LABELS_FILE)

print("Training:", X_train.shape, y_train.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: (7352, 128, 6) (7352,)
Testing: (2947, 128, 6) (2947,)


## Inspect sensor value ranges

In [2]:
print("Training data statistics:\n")

for i, channel in enumerate(CHANNELS):
    values = X_train[:, :, i]

    print(f"{channel}:")
    print(f"  Minimum: {values.min():.4f}")
    print(f"  Maximum: {values.max():.4f}")
    print(f"  Mean:    {values.mean():.4f}")
    print(f"  Std:     {values.std():.4f}")
    print()

Training data statistics:

body_acc_x:
  Minimum: -1.2322
  Maximum: 1.2999
  Mean:    -0.0006
  Std:     0.1948

body_acc_y:
  Minimum: -1.3453
  Maximum: 0.9760
  Mean:    -0.0003
  Std:     0.1224

body_acc_z:
  Minimum: -1.3647
  Maximum: 1.0669
  Mean:    -0.0003
  Std:     0.1069

body_gyro_x:
  Minimum: -4.7337
  Maximum: 4.1555
  Mean:    0.0005
  Std:     0.4068

body_gyro_y:
  Minimum: -5.9743
  Maximum: 5.7461
  Mean:    -0.0008
  Std:     0.3819

body_gyro_z:
  Minimum: -2.7630
  Maximum: 2.3660
  Mean:    0.0001
  Std:     0.2557



## Inspect participant-wise split

In [3]:
train_subjects = np.loadtxt(
    TRAIN_SUBJECTS_FILE,
    dtype=int
)

test_subjects = np.loadtxt(
    TEST_SUBJECTS_FILE,
    dtype=int
)

print("Training participants:", np.unique(train_subjects))
print("Testing participants:", np.unique(test_subjects))

print(
    "Shared participants:",
    np.intersect1d(
        np.unique(train_subjects),
        np.unique(test_subjects)
    )
)

Training participants: [ 1  3  5  6  7  8 11 14 15 16 17 19 21 22 23 25 26 27 28 29 30]
Testing participants: [ 2  4  9 10 12 13 18 20 24]
Shared participants: []


## Convert labels
Our current activity labels are: 1 2 3 4 5 6
Expected PyTorch labels: 0 1 2 3 4 5

In [5]:
# Convert activity labels from 1-6 to 0-5

y_train_encoded = y_train - 1
y_test_encoded = y_test - 1

print("Original training labels:", np.unique(y_train))
print("Encoded training labels:", np.unique(y_train_encoded))

print("Original testing labels:", np.unique(y_test))
print("Encoded testing labels:", np.unique(y_test_encoded))

Original training labels: [1 2 3 4 5 6]
Encoded training labels: [0 1 2 3 4 5]
Original testing labels: [1 2 3 4 5 6]
Encoded testing labels: [0 1 2 3 4 5]


## Verify label distribution

In [6]:
print("Training label distribution:")

unique_labels, counts = np.unique(
    y_train_encoded,
    return_counts=True
)

for label, count in zip(unique_labels, counts):
    print(f"{label}: {count}")

print("\nTesting label distribution:")

unique_labels, counts = np.unique(
    y_test_encoded,
    return_counts=True
)

for label, count in zip(unique_labels, counts):
    print(f"{label}: {count}")

Training label distribution:
0: 1226
1: 1073
2: 986
3: 1286
4: 1374
5: 1407

Testing label distribution:
0: 496
1: 471
2: 420
3: 491
4: 532
5: 537


## Prepare signal dimensions for a 1D CNN

Our original signal shape is:  
(samples, time_steps, channels)  
(7352, 128, 6)  


However, PyTorch's nn.Conv1d expects:  
(batch_size, channels, sequence_length)  


Therefore, we need to rearrange the dimensions:

In [7]:
# Rearrange dimensions for PyTorch Conv1d

X_train_cnn = np.transpose(X_train, (0, 2, 1))
X_test_cnn = np.transpose(X_test, (0, 2, 1))

print("Training CNN input shape:", X_train_cnn.shape)
print("Testing CNN input shape:", X_test_cnn.shape)

Training CNN input shape: (7352, 6, 128)
Testing CNN input shape: (2947, 6, 128)


## Verify the prepared data

In [8]:
print("Training signals:", X_train_cnn.shape)
print("Training labels:", y_train_encoded.shape)

print("Testing signals:", X_test_cnn.shape)
print("Testing labels:", y_test_encoded.shape)

print("\nData validation:")

print("Training signals finite:", np.isfinite(X_train_cnn).all())
print("Testing signals finite:", np.isfinite(X_test_cnn).all())

print("Training labels valid:", np.all(
    (y_train_encoded >= 0) & (y_train_encoded < 6)
))

print("Testing labels valid:", np.all(
    (y_test_encoded >= 0) & (y_test_encoded < 6)
))

Training signals: (7352, 6, 128)
Training labels: (7352,)
Testing signals: (2947, 6, 128)
Testing labels: (2947,)

Data validation:
Training signals finite: True
Testing signals finite: True
Training labels valid: True
Testing labels valid: True
